<img src="https://wamugii.kento.rw/logo-emblem-128.png" width="56" align="right" alt="WAMUGII">

**Deep Learning & Neural Networks** · Week 4 · Sequences, Attention and Generative Models · Lesson 20 of 24

# Recurrent Networks and LSTMs for Sequences

**How to use this notebook:** run the code cells **in order**, from top to bottom (press ▶ on each cell, or use **Runtime → Run all**). You can change any code and run it again. To keep your changes, use **File → Save a copy in Drive**.

**This lesson uses a GPU.** Before you start: **Runtime → Change runtime type → T4 GPU → Save**. The code also runs without a GPU, only more slowly.

The lesson, quiz and certificate are on [WAMUGII](https://wamugii.kento.rw/courses/deep-learning-neural-networks).

**Learning objective:** Explain how recurrent networks carry memory through a sequence, why LSTMs were invented, and forecast a time series with an LSTM compared with honest baselines.

## Data that comes in order

Sales per day, rainfall per week, words in a sentence, heartbeats in an ECG — in a **sequence**, order carries meaning. A **recurrent neural network (RNN)** reads one step at a time and keeps a **hidden state**: a running summary of everything it has seen so far.

> new hidden state = activation(W × current input + U × previous hidden state + b)

The same weights are used at every step — like weight sharing in a CNN, but along time.

## The trouble with plain RNNs

Training an RNN means backpropagating through every time step — a very deep network. Gradients vanish (or explode) over long sequences, so a plain RNN forgets what happened more than a few steps ago.

The **LSTM** (Long Short-Term Memory, 1997) adds a **cell state** — a conveyor belt of memory — controlled by three **gates**:

| Gate | Decides |
|---|---|
| Forget gate | What to erase from memory |
| Input gate | What new information to store |
| Output gate | What to reveal as the hidden state |

Gates are small sigmoid layers whose outputs (0 to 1) act like taps. The **GRU** is a simpler cousin with two gates. Before transformers (2017), LSTMs powered speech recognition and machine translation.

## The data: daily electricity demand

We simulate two years of daily electricity demand (in megawatts) for a city: a weekly pattern (lower on weekends), slow growth, a yearly wave and noise.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

rng = np.random.default_rng(4)
days = np.arange(730)
weekly = np.array([1.0, 1.02, 1.03, 1.03, 1.05, 0.92, 0.85])[days % 7]
demand = (150 + 0.04 * days) * weekly + 8 * np.sin(2 * np.pi * days / 365) + rng.normal(0, 3, len(days))

plt.plot(demand)
plt.xlabel("day")
plt.ylabel("demand (MW)")
plt.title("Simulated daily electricity demand")
plt.show()

## Windows: turn a series into examples

Each example is the last **28 days**; the target is the **next** day. We train on the first 600 days and test on the rest — never shuffle time series across the split, or the model sees the future.

In [ ]:
WINDOW = 28
mean, std = demand[:600].mean(), demand[:600].std()      # training statistics only
z = (demand - mean) / std

X = np.stack([z[i:i + WINDOW] for i in range(len(z) - WINDOW)])
y = z[WINDOW:]
split = 600 - WINDOW
X_train, y_train, X_test, y_test = X[:split], y[:split], X[split:], y[split:]
real = y_test * std + mean
print(X_train.shape, X_test.shape)

## Baselines first

In [ ]:
yesterday = X_test[:, -1] * std + mean          # "tomorrow = today"
last_week = X_test[:, -7] * std + mean          # "tomorrow = same day last week"
print("MAE, same as yesterday:", round(np.abs(yesterday - real).mean(), 2), "MW")
print("MAE, same as last week:", round(np.abs(last_week - real).mean(), 2), "MW")

"Same as yesterday" is poor (about **11 MW** off) because of the weekend drop; "same day last week" is strong (about **3.5 MW**). Any model must beat **3.5**.

## An LSTM forecaster

In [ ]:
class Forecaster(nn.Module):
    def __init__(self, hidden=32):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden, batch_first=True)
        self.out = nn.Linear(hidden, 1)

    def forward(self, x):                    # x: (batch, 28 days)
        outputs, (h, c) = self.lstm(x.unsqueeze(-1))
        return self.out(outputs[:, -1]).squeeze(-1)   # use the state after the last day

t = lambda a: torch.tensor(a, dtype=torch.float32)
torch.manual_seed(0)
model = Forecaster()
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
for epoch in range(60):
    model.train()
    perm = torch.randperm(len(X_train))
    for start in range(0, len(perm), 32):
        idx = perm[start:start + 32]
        opt.zero_grad()
        F.mse_loss(model(t(X_train[idx])), t(y_train[idx])).backward()
        opt.step()

model.eval()
with torch.no_grad():
    forecast = model(t(X_test)).numpy() * std + mean
print("MAE, LSTM:", round(np.abs(forecast - real).mean(), 2), "MW")

plt.plot(real, label="real")
plt.plot(forecast, label="LSTM forecast")
plt.legend()
plt.title("Test period: next-day forecasts")
plt.show()

The LSTM is off by about **2.9 MW**, beating the strong weekly baseline (about 3.5). It learned the weekly rhythm **and** the slower changes from the data, with no calendar given to it.

## Sequences beyond time series

- `nn.LSTM` also reads **text**: feed it the word embeddings from Lesson 19 instead of one number per day.
- **Bidirectional** LSTMs read forwards and backwards (useful for whole sentences).
- For many sequence tasks today, **transformers** (next lesson) are faster to train and more accurate, but LSTMs remain useful for small data and streaming signals on small devices.

## Key terms

- **Sequence** — data where order matters.
- **RNN** — a network that reads step by step and keeps a hidden state.
- **Hidden state** — the running memory of an RNN.
- **LSTM / GRU** — RNNs with gates that keep long-term memory.
- **Window** — a fixed number of past steps used as one example.
- **Seasonal baseline** — "same as last week": the forecast to beat.

**Takeaway:** RNNs and LSTMs carry a memory along a sequence; for forecasting, build windows without peeking at the future and always beat a seasonal baseline.

## Try it yourself

1. Replace `nn.LSTM` with `nn.GRU`. Is it better, worse or the same?
2. Use a 14-day window instead of 28. What happens to the error, and why?
3. Forecast 7 days ahead instead of 1 (change the target). How much harder is it?
4. Add the day of the week as a second input feature (`input_size=2`). Does the LSTM improve?

---
Finished? Go back to [Deep Learning & Neural Networks on WAMUGII](https://wamugii.kento.rw/courses/deep-learning-neural-networks), tick the lesson as done and take the quiz.

*© WAMUGII TECH SOLUTIONS · Original course material for WAMUGII AI Program students.*